In [1]:
import xarray as xr
import numpy as np



In [2]:

def write_cdo_curvilinear_grid(
    nc_file,
    out_grid_file,
    lon_name="lon",
    lat_name="lat",
    add_bounds=False,
):
    ds = xr.open_dataset(nc_file)

    if lon_name not in ds.variables:
        raise ValueError(f"Longitude variable '{lon_name}' not found")
    if lat_name not in ds.variables:
        raise ValueError(f"Latitude variable '{lat_name}' not found")

    lon = ds[lon_name].values
    lat = ds[lat_name].values

    if lon.ndim != 2 or lat.ndim != 2:
        raise ValueError("lon and lat must both be 2D arrays")

    if lon.shape != lat.shape:
        raise ValueError("lon and lat must have the same shape")

    ny, nx = lon.shape

    def format_array(arr):
        flat = arr.ravel()
        return "\n".join(
            " ".join(f"{x:.10f}" for x in flat[i:i+6])
            for i in range(0, len(flat), 6)
        )

    with open(out_grid_file, "w") as f:
        f.write("gridtype = curvilinear\n")
        f.write(f"xsize    = {nx}\n")
        f.write(f"ysize    = {ny}\n")
        f.write(f"gridsize = {nx * ny}\n")
        f.write("xname    = lon\n")
        f.write('xlongname = "longitude"\n')
        f.write('xunits   = "degrees_east"\n')
        f.write("yname    = lat\n")
        f.write('ylongname = "latitude"\n')
        f.write('yunits   = "degrees_north"\n')
        f.write("xvals    = \n")
        f.write(format_array(lon))
        f.write("\n")
        f.write("yvals    = \n")
        f.write(format_array(lat))
        f.write("\n")

    ds.close()
    print(f"Wrote CDO grid file: {out_grid_file}")


# example usage
write_cdo_curvilinear_grid(
    nc_file="/wolke_scratch/dnikolo/Glaciation_time_estimator/grids/agg_pixel_area_np_firstslice.nc",
    out_grid_file="/wolke_scratch/dnikolo/ICON_Data/aux_tools/target_curvilinear_grid.txt",
    lon_name="lon",
    lat_name="lat",
)

Wrote CDO grid file: /wolke_scratch/dnikolo/ICON_Data/aux_tools/target_curvilinear_grid.txt


## With bounds

In [ ]:



def _infer_vertices_from_centers(arr):
    """
    Infer node/vertex values from cell-center values.
    Input shape:  (ny, nx)
    Output shape: (ny+1, nx+1)
    """
    ny, nx = arr.shape
    v = np.empty((ny + 1, nx + 1), dtype=np.float64)

    # interior nodes: average of 4 surrounding centers
    v[1:ny, 1:nx] = 0.25 * (
        arr[:-1, :-1] + arr[:-1, 1:] + arr[1:, :-1] + arr[1:, 1:]
    )

    # edges: linear extrapolation from nearest centers
    v[0, 1:nx] = 0.5 * (arr[0, :-1] + arr[0, 1:])
    v[ny, 1:nx] = 0.5 * (arr[ny - 1, :-1] + arr[ny - 1, 1:])
    v[1:ny, 0] = 0.5 * (arr[:-1, 0] + arr[1:, 0])
    v[1:ny, nx] = 0.5 * (arr[:-1, nx - 1] + arr[1:, nx - 1])

    # corners: simple extrapolation
    v[0, 0] = arr[0, 0] - 0.5 * (arr[1, 1] - arr[0, 0]) if ny > 1 and nx > 1 else arr[0, 0]
    v[0, nx] = arr[0, nx - 1] - 0.5 * (arr[1, nx - 2] - arr[0, nx - 1]) if ny > 1 and nx > 1 else arr[0, nx - 1]
    v[ny, 0] = arr[ny - 1, 0] - 0.5 * (arr[ny - 2, 1] - arr[ny - 1, 0]) if ny > 1 and nx > 1 else arr[ny - 1, 0]
    v[ny, nx] = arr[ny - 1, nx - 1] - 0.5 * (arr[ny - 2, nx - 2] - arr[ny - 1, nx - 1]) if ny > 1 and nx > 1 else arr[ny - 1, nx - 1]

    return v


def centers_to_bounds(lon, lat):
    """
    Convert 2D center lon/lat to 4-corner bounds per cell.
    Returns:
      lon_bnds, lat_bnds with shape (ny, nx, 4)
    Corner order: SW, SE, NE, NW
    """
    lon_vertices = _infer_vertices_from_centers(lon)
    lat_vertices = _infer_vertices_from_centers(lat)

    ny, nx = lon.shape
    lon_bnds = np.empty((ny, nx, 4), dtype=np.float64)
    lat_bnds = np.empty((ny, nx, 4), dtype=np.float64)

    # SW
    lon_bnds[:, :, 0] = lon_vertices[:-1, :-1]
    lat_bnds[:, :, 0] = lat_vertices[:-1, :-1]

    # SE
    lon_bnds[:, :, 1] = lon_vertices[:-1, 1:]
    lat_bnds[:, :, 1] = lat_vertices[:-1, 1:]

    # NE
    lon_bnds[:, :, 2] = lon_vertices[1:, 1:]
    lat_bnds[:, :, 2] = lat_vertices[1:, 1:]

    # NW
    lon_bnds[:, :, 3] = lon_vertices[1:, :-1]
    lat_bnds[:, :, 3] = lat_vertices[1:, :-1]

    return lon_bnds, lat_bnds


def _format_array(arr, ncol=6):
    flat = np.asarray(arr).ravel()
    lines = []
    for i in range(0, len(flat), ncol):
        chunk = flat[i:i+ncol]
        lines.append(" ".join(f"{v:.10f}" for v in chunk))
    return "\n".join(lines)


def write_cdo_curvilinear_grid_with_bounds(
    nc_file,
    out_grid_file,
    lon_name="lon",
    lat_name="lat",
):
    ds = xr.open_dataset(nc_file)

    lon = ds[lon_name].values
    lat = ds[lat_name].values

    while lon.ndim > 2:
        lon = lon[0]
    while lat.ndim > 2:
        lat = lat[0]

    if lon.ndim != 2 or lat.ndim != 2:
        raise ValueError("lon and lat must be 2D after squeezing")

    if lon.shape != lat.shape:
        raise ValueError("lon and lat must have the same shape")

    ny, nx = lon.shape
    lon_bnds, lat_bnds = centers_to_bounds(lon, lat)

    with open(out_grid_file, "w") as f:
        f.write("gridtype = curvilinear\n")
        f.write(f"xsize    = {nx}\n")
        f.write(f"ysize    = {ny}\n")
        f.write(f"gridsize = {nx * ny}\n")
        f.write("xname    = lon\n")
        f.write('xlongname = "longitude"\n')
        f.write('xunits   = "degrees_east"\n')
        f.write("yname    = lat\n")
        f.write('ylongname = "latitude"\n')
        f.write('yunits   = "degrees_north"\n')

        f.write("xvals    = \n")
        f.write(_format_array(lon))
        f.write("\n")
        f.write("yvals    = \n")
        f.write(_format_array(lat))
        f.write("\n")

        f.write("nvertex  = 4\n")
        f.write("xbounds  = \n")
        f.write(_format_array(lon_bnds))
        f.write("\n")
        f.write("ybounds  = \n")
        f.write(_format_array(lat_bnds))
        f.write("\n")

    ds.close()
    print(f"Wrote {out_grid_file}")

# example usage
write_cdo_curvilinear_grid_with_bounds(
    nc_file="/wolke_scratch/dnikolo/Glaciation_time_estimator/grids/agg_pixel_area_np_firstslice.nc",
    out_grid_file="/wolke_scratch/dnikolo/ICON_Data/aux_tools/grid_CLAAS_Agg_3_np.txt",
    lon_name="lon",
    lat_name="lat",
)

Wrote /wolke_scratch/dnikolo/ICON_Data/aux_tools/grid_CLAAS_Agg_3_np.txt


In [1]:
-0.06148785800938583*3

-0.1844635740281575